# StatBridge Golden v2 구현 검토

**ai-assisted / 공개 자료 전용 / 사람 승인 전 초안**

이 노트북은 이미 작성한 72건과 로컬 실행 관측을 읽어 확인합니다. 외부 API를 호출하지 않으며 정답·승인 상태를 수정하지 않습니다.

읽는 순서: 구성 → 최신 자체 검증·보완 보고서 → 실제 채점 → 사례 선택 → 데이터·그림·출처 비교 → 실패 원인 → 구현 코드.
기본 관측은 보완 후 새로 실행한 대표 12건입니다. 나머지 60건은 이 실행에서 미실행으로 표시하며 이전 dev/test 관측과 합치지 않습니다.

- 정답 fixture의 참고 그래프와 **실제 앱이 반환한 Plotly 그래프**를 구분합니다.
- `pending_review`는 성공이 아닙니다. 오프라인 관측은 실서비스 검증이 아닙니다.
- 결과 파일이 없으면 미실행으로 표시합니다. 숫자를 만들어 채우지 않습니다.
- Git에 올릴 때는 **모든 셀 출력 제거** 후 저장하세요. 예측·평가 결과는 `results/`에서만 관리합니다.


In [ ]:
from pathlib import Path
from collections import Counter
import hashlib
import json
import sys
import pandas as pd

# 다른 위치에서 Jupyter를 실행했다면 프로젝트 루트를 지정하세요.
PROJECT_ROOT = ""
SHOW_GRAPHS = True

if PROJECT_ROOT:
    ROOT = Path(PROJECT_ROOT).resolve()
else:
    ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / "eval/statbridge-golden/v2/dev.jsonl").exists()), None)
    if ROOT is None:
        raise RuntimeError("PROJECT_ROOT에 StatBridge-local 경로를 지정하세요.")
BASE = ROOT / "eval/statbridge-golden/v2"
sys.path.insert(0, str(BASE / "scripts"))

try:
    from IPython.display import display, Markdown, Code
except ImportError:
    # 커널 없이 코드 셀을 검증하는 경우의 텍스트 출력.
    display = print
    Markdown = str
    def Code(data, language="python"):
        return data

from common import read_json, read_jsonl, digest, identity
from validate import validate_cases
from score import score, unique_rows
from run import observation_hash

cases = read_jsonl(BASE / "dev.jsonl") + read_jsonl(BASE / "test.jsonl")
case_by_id = unique_rows(cases, "case")
display(pd.DataFrame([{
    "프로젝트": str(ROOT), "Python": sys.executable, "공개 사례": len(cases),
    "사람 승인": sum(c["review"]["human_approved"] for c in cases),
    "평가 환경": "오프라인 진단 / 공식 평가 아님"
}]))


## 1. 골든셋 구성과 출처

네 모드는 평가하는 단계가 다릅니다. 같은 사례 수라도 점수를 서로 합쳐 공식 성능으로 표시하면 안 됩니다.

연구 후보 5종의 분포와 실제 채택한 보고서 분포를 나란히 확인합니다. 후보 수는 검증된 원본 수가 아닙니다.


In [ ]:
catalog = pd.DataFrame([{
    "id": c["id"], "split": c["split"], "mode": c["mode"], "group_id": c["group_id"],
    "tags": ", ".join(c["tags"]), "질문": c["input"]["query"],
    "사람 승인": c["review"]["human_approved"], "검수 상태": c["review"]["status"]
} for c in cases])
display(pd.crosstab(catalog["mode"], catalog["split"], margins=True))
inventory = read_json(BASE / "source_inventory.json")
display(pd.DataFrame([
    {"보고서": name, "연구 후보": count,
     "채택 고유 결과물": inventory["selected_unique_result_ids_by_report_type"].get(name, 0)}
    for name, count in inventory["candidate_report_types"].items()
]))
display(pd.DataFrame(read_json(BASE / "candidate_probes.json"))[
    ["report_type", "title", "title_observed", "selected", "reason"]
])


## 2. 계약·해시 검증

검증기는 스키마, 표·항목·분류 ID, 주기·단위, 중복, 그룹 분리, fixture 및 원본 해시를 검사합니다.

원본이 있으면 해시를 비교합니다. 기본 False에서는 원본 누락을 경고로 표시합니다. `VERIFY_ORIGINALS=True`로 바꾸면 모든 원본의 존재까지 필수로 검사합니다. 기계 검증 통과가 사람의 정답 승인은 아닙니다.


In [ ]:
VERIFY_ORIGINALS = False
validation = validate_cases(cases, verify_sources=VERIFY_ORIGINALS)
display(pd.DataFrame([{
    "사례": validation["count"], "검증 오류": len(validation["errors"]),
    "원본 경고": len(validation["warnings"]), "사람 승인": validation["human_approved"]
}]))
if validation["errors"]:
    display(pd.DataFrame({"검증 오류": validation["errors"]}))
if validation["warnings"]:
    display(pd.DataFrame({"검증 경고": validation["warnings"]}))

manifest = read_json(BASE / "dataset_manifest.json")
changed_files = [
    relative for relative, sha in manifest["files"].items()
    if not (BASE / relative).exists() or digest(BASE / relative) != sha
]
display(pd.DataFrame({"manifest와 다른 파일": changed_files}))
# 노트북을 편집했다면 노트북 해시 차이는 자연스럽습니다.
# 사례/fixture 변경은 별도의 정답 검수와 manifest 재생성이 필요합니다.

display(Markdown('### 최신 자체 검증과 단위 근거'))
current_hashes = {split: digest(BASE / (split + '.jsonl')) for split in ('dev', 'test')}
for audit_name in ('sources', 'tools'):
    audit_path = BASE / 'results' / ('self-audit-' + audit_name + '.json')
    if not audit_path.exists():
        display(Markdown(audit_name + ': 저장된 자체 검증 없음'))
        continue
    audit = read_json(audit_path)
    bound = audit.get('dataset_hashes') == current_hashes
    display(pd.DataFrame([{'자체 검증': audit_name, '현재 사례 해시 일치': bound,
                           '사람 승인 부여': audit.get('human_approval_granted')}]))
    if not bound:
        display(Markdown('**현재 사례와 다른 검증 기록입니다. 최신 검증으로 해석하지 마세요.**'))
    if audit_name == 'sources':
        display(pd.DataFrame([{k: audit.get(k) for k in ('series_checks', 'point_checks', 'failed_checks')}]))
        display(pd.DataFrame(audit.get('unit_definitions', [])))
        display(pd.DataFrame(audit.get('transformations', [])))
    else:
        display(pd.DataFrame(audit.get('adversarial_probes', [])))
display(pd.json_normalize(read_json(BASE / 'unit_rules.json')['rules']))
report_path = BASE / 'results/UI_REVISION_REPORT.md'
if report_path.exists():
    display(Markdown(report_path.read_text(encoding='utf-8')))
else:
    display(Markdown('저장된 보완 보고서가 없습니다. 규칙은 HARDENING.md에서 확인하세요.'))


## 3. 저장된 실제 관측으로 다시 채점

`OBSERVATION_SET = "ui-dev-test"`은 UI 정합성 수정 후 전체 관측을 읽습니다. `pilot-ui`로 바꾸면 새 대표 12건을 살펴봅니다. 이전 관측과 합치거나 API를 재실행하지 않습니다.

- 자동 통과: 표·데이터·그래프 등의 자동 기준을 충족한 관측
- 해석 검수 대기: 자동 기준을 충족했지만 사람의 문장 검수가 없음
- 실패: 실제 오류 또는 자동 기준 불일치
- 미실행/미관측: 성공으로 포함하지 않음

사람의 골든 정답 승인과 응답 해석 검수는 별개입니다. 아래 수치는 초안 진단입니다.


In [ ]:
OBSERVATION_SET = "ui-dev-test"  # ui-dev-test / pilot-ui
if OBSERVATION_SET not in {'ui-dev-test', 'pilot-ui'}:
    raise ValueError('지원하지 않는 관측 선택입니다.')
run_names = ('pilot-ui',) if OBSERVATION_SET == 'pilot-ui' else ('dev-ui', 'test-ui')
display(Markdown('**선택한 저장 관측: ' + OBSERVATION_SET + ' / 미포함 사례는 미실행**'))
predictions = []
reviews = []
for split in run_names:
    prediction_path = BASE / "results" / (split + ".jsonl")
    review_path = BASE / "results" / (split + "-review.jsonl")
    if prediction_path.exists():
        predictions.extend(read_jsonl(prediction_path))
    else:
        display(Markdown("**" + split + ": 저장된 실행 관측 없음.**"))
    if review_path.exists():
        reviews.extend(read_jsonl(review_path))

prediction_by_id = unique_rows(predictions, "prediction")
bad_observation_hashes = [
    p["id"] for p in predictions
    if p.get("observation_sha256") != observation_hash(p)
]
if bad_observation_hashes:
    raise ValueError("수정되었거나 손상된 관측: " + ", ".join(bad_observation_hashes))

graded = score(cases, predictions, reviews, base=BASE)
grades = pd.DataFrame(graded["cases"])
display(pd.DataFrame(graded["automatic_summary"]).T)
display(pd.crosstab(grades["mode"], grades["status"]))
display(Markdown("**공식 정답 승인 건수: " + str(graded["official_approved_cases"]) + "**"))
display(pd.DataFrame(graded["approved_summary"]).T)


## 4. 사례 목록 필터

아래 변수로 split, 모드, 태그, 실패 여부를 선택합니다. `all`은 전체입니다. 검토할 id를 다음 섹션의 CASE_ID에 입력하세요.


In [ ]:
SPLIT = "all"       # dev / test / all
MODE = "all"        # discovery / e2e / output / edit / all
TAG = ""            # 예: missing, multi, followup
ONLY_FAILURES = False

overview = catalog.merge(grades[["id", "status", "failures"]], on="id", how="left")
filtered = overview.copy()
if SPLIT != "all":
    filtered = filtered[filtered["split"] == SPLIT]
if MODE != "all":
    filtered = filtered[filtered["mode"] == MODE]
if TAG:
    matching = {c["id"] for c in cases if TAG in c["tags"]}
    filtered = filtered[filtered["id"].isin(matching)]
if ONLY_FAILURES:
    filtered = filtered[filtered["status"] == "failed"]
with pd.option_context("display.max_rows", 72, "display.max_colwidth", 100):
    display(filtered[["id", "split", "mode", "tags", "status", "failures", "질문"]])


## 5. 한 사례의 질문·정답·관측 비교

기본 사례 `SBV2-0061`은 실제 조회 기반 금리 출력입니다. 첨부 자료·업로드를 가정하지 않습니다.

`input`만 프로그램에 전달됩니다. `expected`는 채점용이며 세션 ID를 정답에서 채우지 않습니다.


In [ ]:
CASE_ID = "SBV2-0061"
if CASE_ID not in case_by_id:
    raise ValueError("공개 v2에 없는 CASE_ID입니다.")
case = case_by_id[CASE_ID]
prediction = prediction_by_id.get(CASE_ID)
case_grade = grades.loc[grades["id"] == CASE_ID].iloc[0]
display(Markdown("### " + CASE_ID + " / " + case["mode"] + " / " + case["split"]))
display(Markdown(case["input"]["query"]))
display(Code(json.dumps(case["input"], ensure_ascii=False, indent=2), language="json"))
display(Code(json.dumps(case["expected"].get("resolution", {}), ensure_ascii=False, indent=2), language="json"))
display(pd.DataFrame([{
    "실행 상태": prediction.get("execution_status") if prediction else "not_executed",
    "평가 상태": case_grade["status"], "실패 코드": case_grade["failures"],
    "실행 오류": prediction.get("error") if prediction else None,
    "관측 환경": prediction.get("environment") if prediction else None,
    "수정 분류기": prediction.get("edit_parser") if prediction else None
}]))
if prediction and prediction.get("resolution"):
    display(Code(json.dumps(prediction["resolution"], ensure_ascii=False, indent=2), language="json"))


## 6. 시점·숫자·결측·계열 식별자

표·항목·분류·주기·단위 또는 attachment series_id까지 같아야 같은 계열로 비교합니다.

정답이나 실제 값이 없는 자리를 0으로 채우지 않습니다. 정답 null과 실제 시점 누락도 다릅니다.


In [ ]:
expected_series = (read_json(BASE / case["expected"]["data"]["fixture"])["series"]
                   if "data" in case["expected"] else [])
actual_series = prediction.get("series", []) if prediction else []

def series_frame(series):
    return pd.DataFrame([{
        "계열 식별자": repr(identity(s)), "계열명": s["label"], "단위": s["unit"],
        "주기": s["frequency"], "시점": p["period"], "값": p["value"], "명시적 결측": p["value"] is None
    } for s in series for p in s["points"]],
    columns=["계열 식별자", "계열명", "단위", "주기", "시점", "값", "명시적 결측"])

expected_df = series_frame(expected_series)
actual_df = series_frame(actual_series)
display(Markdown("**정답 fixture**"))
display(expected_df)
display(Markdown("**실제 관측 데이터**"))
display(actual_df)

comparison = expected_df.merge(actual_df, on=["계열 식별자", "시점"],
                              how="outer", suffixes=("_정답", "_실제"), indicator=True)
comparison["차이(실제-정답)"] = comparison["값_실제"] - comparison["값_정답"]
display(comparison)


## 7. 정답 참고도와 실제 Plotly 결과

첫 그림은 fixture를 사람이 읽기 쉽게 직접 그린 **참고도**입니다. 제품이 생성한 출력이 아닙니다.
두 번째 그림은 저장된 실제 앱의 `plotly_figure`를 그대로 표시합니다.

숫자·시점·trace·subplot 검사와 사람의 해석 판정은 별도로 진행합니다.


In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def reference_figure(series, output):
    separate = output.get("layout") == "separate" and len(series) > 1
    fig = make_subplots(rows=len(series), cols=1, subplot_titles=[s["label"] for s in series]) if separate else go.Figure()
    chart_type = output.get("acceptable_chart_types", ["line"])[0]
    for i, s in enumerate(series):
        x = [p["period"] for p in s["points"]]
        y = [p["value"] for p in s["points"]]
        trace = (go.Bar(x=x, y=y, name=s["label"]) if chart_type in {"bar", "stacked_bar"}
                 else go.Scatter(x=x, y=y, name=s["label"], mode="lines+markers", connectgaps=False))
        if separate:
            fig.add_trace(trace, row=i+1, col=1)
        else:
            fig.add_trace(trace)
    fig.update_layout(title="정답 fixture 참고도 / 서비스 출력 아님",
                      height=max(360, 220*len(series)) if separate else 420)
    return fig

if SHOW_GRAPHS:
    if expected_series:
        reference_figure(expected_series, case["expected"].get("output", {})).show()
    else:
        display(Markdown("이 discovery 사례에는 수치 그래프 정답이 없습니다."))
    if prediction and prediction.get("plotly_figure"):
        display(Markdown("**실제 앱 Plotly 출력 / " + prediction["environment"] + "**"))
        go.Figure(prediction["plotly_figure"]).show()
    else:
        display(Markdown("실제 그래프 관측이 없습니다. 정답 그래프로 대신하지 않습니다."))


## 8. 출처와 해석 검수

`purpose_reconstruction`은 원본 그림의 주제에 관련된 지원 계열로 만든 새 질문입니다. 원본 그림의 수치·전망을 정확히 복제했다는 뜻이 아닙니다.

필수 사실과 실제 설명을 함께 읽고 단위·방향·모순·근거 없는 원인 단정을 확인하세요. 여기서 승인 값을 자동으로 변경하지 않습니다.


In [ ]:
display(Markdown("**근거 관계: " + case["evidence"]["relationship"] + "**"))
display(pd.DataFrame(case["evidence"]["sources"])[["path", "locator", "sha256", "publisher", "extraction"]])
display(pd.DataFrame({"research 결과물 ID": case["evidence"]["research_result_ids"]}))
if "output" in case["expected"]:
    claims = read_json(BASE / case["expected"]["output"]["claims"])
    display(pd.json_normalize(claims["facts"]))
if prediction:
    display(Markdown("**실제 응답의 설명**"))
    display(Code(json.dumps(prediction.get("explanation"), ensure_ascii=False, indent=2), language="json"))
display(pd.DataFrame([case["review"]]))


## 9. API 단계와 실패 코드

실패 코드는 증상입니다. 원본·입력·실제 응답을 확인한 뒤 골든 정답 문제와 제품 결함을 구분해야 합니다.
API 오류 뒤에 그래프 검사 실패가 여러 개 따라올 수 있으므로 먼저 실행 오류를 살펴보세요.


In [ ]:
trace = prediction.get("trace", []) if prediction else []
display(pd.DataFrame([{
    "단계": t["stage"], "HTTP": t.get("http_status"), "시간(ms)": t.get("elapsed_ms"),
    "응답 상태": t.get("response", {}).get("status")
} for t in trace]))
display(pd.DataFrame(prediction.get("statistics_requests", []) if prediction else []))

failure_help = {
    "execution_error": "실행 단계에서 오류가 발생했습니다. error와 trace의 마지막 단계를 먼저 확인하세요.",
    "selected_tables": "완전한 허용 표 집합과 실제 선택이 다릅니다.",
    "selected_series_identity": "항목·분류·주기·단위까지 고정한 계열 선택이 다릅니다.",
    "resolution_decision": "표 확정·역질문·미지원 중 판정이 다릅니다.",
    "series_identity_or_count": "계열 ID/항목/분류/단위/주기 또는 개수가 다릅니다.",
    "periods_or_order": "시점 누락·추가·순서 차이가 있습니다. null이 삭제되었는지도 확인하세요.",
    "numeric_value": "값이 없거나 사례에 지정된 숫자 오차를 넘었습니다.",
    "missing_value_imputed": "원본 결측이 다른 값으로 바뀌었습니다.",
    "missing_point_value": "값 필드 자체가 없습니다. 명시적인 null과 다릅니다.",
    "input_hash_missing": "실행 입력을 확인할 해시가 없습니다.",
    "input_hash_mismatch": "관측이 현재 사례 입력과 연결되지 않습니다.",
    "plotly_line_mode": "선그래프에 실제 선을 표시하는 모드가 없습니다.",
    "plotly_hidden_line": "선이 숨겨지거나 두께·불투명도가 0입니다.",
    "clarification_semantics_unreviewed": "단어 포함만으로 선택지 의미를 확정할 수 없어 사람 검수가 필요합니다.",
    "clarification_options": "필요한 역질문 선택지가 빠졌거나 다릅니다.",
    "clarification_dimensions": "질문에서 필요한 확인 기준을 표현하지 않았습니다.",
    "plotly_values": "실제 Plotly x/y가 실제 계열 데이터와 다릅니다.",
    "chart_layout": "합쳐 보기 / 나누어 보기 배치가 정답과 다릅니다.",
    "edit_changed_source_data": "수정 중 유지해야 할 원자료·출처가 바뀌었습니다."
}
counts = Counter(f for row in graded["cases"] for f in row.get("failures", []))
display(pd.DataFrame([{"실패 코드": key, "건수": count,
    "설명": failure_help.get(key, "scripts/score.py의 해당 검사를 확인하세요.")}
    for key, count in counts.most_common()]))


## 10. 실제 구현 코드 읽기

`MODULE`을 바꾸면 제작·검증·실행·채점 구현을 읽을 수 있습니다.
서비스 API나 정답을 수정하는 셀이 아닙니다. 검수 절차는 `review/GUIDE.md`에 있습니다.


In [ ]:
MODULE = "score.py"  # author.py / validate.py / run.py / common.py / review_packet.py
allowed_modules = {"author.py", "validate.py", "run.py", "common.py", "score.py", "review_packet.py", "edit_contract.py", "self_audit.py"}
if MODULE not in allowed_modules:
    raise ValueError("공개 v2 검토 대상 모듈만 선택하세요.")
display(Code((BASE / "scripts" / MODULE).read_text(encoding="utf-8"), language="python"))


## 선택: 대표 12건을 다시 실행하기

기본값은 False입니다. True로 바꾸면 실제 앱의 오프라인 경로를 다시 실행하며 시간이 걸립니다.
외부 HTTP는 차단되고 수정 분류기는 사용자 입력 명령 재생입니다. live/NCP 성능을 측정하지 않습니다.
새 관측은 무시되는 `results/notebook-pilot.jsonl`에만 저장하고 기존 dev/test 결과는 덮어쓰지 않습니다.


In [ ]:
RUN_PILOT = False
if RUN_PILOT:
    from run import run_cases
    from common import write_jsonl
    pilot_cases = read_jsonl(BASE / "pilot.jsonl")
    new_predictions = run_cases(pilot_cases, live=False)
    write_jsonl(BASE / "results/notebook-pilot.jsonl", new_predictions)
    pilot_grade = score(pilot_cases, new_predictions, base=BASE)
    display(pd.DataFrame(pilot_grade["cases"])[["id", "mode", "status", "failures"]])
else:
    display(Markdown("재실행하지 않았습니다. 위 내용은 저장된 공개 관측 검토입니다."))
